First of all, set the 'CODE_DIR' to where the model code are saved. This will change current working directory and print for checking. Afterwards, we import all required modules.

In [1]:
import os

# Save the current PATH
original_path = os.environ['PATH']

# Set CUDA 12.5 environment variables, appending the original PATH explicitly
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.5'
os.environ['PATH'] = f"/usr/local/cuda-12.5/bin:{original_path}"
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-12.5/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

#!rm -rf /home/ids/yuhe/.cache/torch_extensions
CODE_DIR = '/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/pSp_encoder_constructive'

import os
os.chdir(f'{CODE_DIR}')

notebook_path = os.getcwd()
print('Current working directory is:', '\n', notebook_path) 


Current working directory is: 
 /home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/pSp_encoder_constructive


In [2]:
# load psp and cmlp
import torch
from argparse import Namespace
from models.psp import pSp
from models.mlp3D import MappingNetwork_cs_independent, EqualizedLinear
from models.c2s_mlp import MappingNetwork_c2s
from notebooks.load_dataset import get_dataset_loaders
import csv
def init_pretrained_models(pretrained_ckpt_path, device):

    model_ckpt = torch.load(pretrained_ckpt_path, map_location=device, weights_only=True)
    cs_ckpt  = model_ckpt['state_dict_cs_enc']
    model_opts = model_ckpt['opts']
    model_opts = Namespace(**model_opts)

    # load pSp model and pretrained weights
    pSp_net = pSp(model_opts).to(device)

    # load csmlp model and pretrained weights
    cs_mlp_net = MappingNetwork_cs_independent(model_opts).to(device)
    print('Loading csmlp from path: {}'.format(model_opts.exp_dir))     
    cs_mlp_net.load_state_dict(cs_ckpt)

    return pSp_net, cs_mlp_net, model_opts


def calc_Frob_dist_test_dataset(pSp_net, cs_mlp_net, c2s_mlp, test_t_dataloader, opts, device, csv_name, text_name):
    # Main computation loop
    dist_list = []

    # Ensure all models are in evaluation mode
    pSp_net.eval()
    cs_mlp_net.eval()
    c2s_mlp.eval()

    for batch_idx, batch_t in enumerate(test_t_dataloader):  # Loop through test_t_dataloader
        # Move the batch to the specified device
        x_t, _ = batch_t
        x_t = x_t.to(device).float()
        
        with torch.no_grad():
            # Forward pass through pSp_net to get latent codes
            w_pSp = pSp_net.forward(x_t, encode_only=True)
            
            # Decompose latent codes into c_t and s_t using cs_mlp_net
            c_t, s_t = cs_mlp_net(w_pSp, zero_out_silent=opts.zero_out_silent_t)         
            
            # Reconstruct s_t from c_t using c2s_mlp
            s_t_recon = c2s_mlp(c_t)
            
            # Compute Frobenius norm for each 18x512 matrix in the batch
            fro_dist_batch = torch.norm(s_t_recon - s_t, p='fro', dim=(1, 2))
        
        # Append distances for each image in the batch to the dist_list
        dist_list.extend(fro_dist_batch.cpu().tolist())  # Convert distances to a Python list and add to dist_list

    # Calculate the mean of all distances
    mean_dist = torch.tensor(dist_list).mean().item()

    # Save dist_list to a CSV file
    with open(csv_name, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Distance"])  # Header
        for dist in dist_list:
            writer.writerow([dist])

    # Save mean_dist to a text file
    with open(text_name, 'a') as txtfile:
        txtfile.write(f"\n{opts.exp_dir}\n")
        txtfile.write(f"Mean Frobenius Distance: {mean_dist:.4f}\n")

    # Print summary
    print(f"Calculated Frobenius distances for {len(dist_list)} images.")
    print(f"Saved dist_list to {csv_name}")
    print(f"Saved mean Frobenius distance to {text_name}")

    return dist_list, mean_dist



In [3]:
#load pSp-csmlp ckpt 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
csmlp_checkpoint_path='results/csmlp_ffhq_glasses/mlp3D/nodim/checkpoints/iteration_130000.pt'
pSp_net, cs_mlp_net, model_opts = init_pretrained_models(csmlp_checkpoint_path, device)

#load c2s_mlp ckpt
model_opts.dataset_type = 'ffhq_glasses'
_, _, _, test_t_dataloader = get_dataset_loaders(model_opts)

Loading pSp from checkpoint: ../pretrained_models/pSp_models/psp_ffhq_encode.pt
Loading csmlp from path: results/csmlp_sparsity/mlp3D/nodim/
Loading dataset for ffhq_glasses
Number of traing bg samples: 10000
Number of traing t samples: 10000
Number of test bg samples: 3500
Number of test t samples: 3500


In [4]:
#load c2s_mlp ckpt 
folder_name = "10layers_l2_lr0.01"
c2s_ckpt_path = f"./results/train_c2smlp/{folder_name}/checkpoints/iteration_300000.pt"
c2s_ckpt = torch.load(c2s_ckpt_path, map_location=device, weights_only=True)
opts = c2s_ckpt['opts']
opts = Namespace(**opts)
c2s_mlp = MappingNetwork_c2s(opts).to(device)
print('Loading csmlp from path: {}'.format(opts.exp_dir)) 
c2s_mlp.load_state_dict(c2s_ckpt['state_dict_c2s_mlp'])

dist_list, mean_dist = calc_Frob_dist_test_dataset(
    pSp_net=pSp_net,
    cs_mlp_net=cs_mlp_net,
    c2s_mlp=c2s_mlp,
    test_t_dataloader=test_t_dataloader,
    opts=opts,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    csv_name=f"{folder_name}.csv",
    text_name="mean_distance.txt"
)


Loading csmlp from path: results/train_c2smlp/10layers_l2_lr0.01
Calculated Frobenius distances for 3500 images.
Saved dist_list to 10layers_l2_lr0.01.csv
Saved mean Frobenius distance to mean_distance.txt
